# Fake Review Classification — Updated

Perubahan utama dibanding notebook sebelumnya:
- Pseudo-label opsional tapi **dibalance** supaya tidak membuat train jadi bias (Real >> Fake).
- Baseline diperbaiki (tuning sederhana untuk SVM/LogReg, NB pakai alpha).
- 3 model TensorFlow (DNN, Wide&Deep, ResDNN) memakai input **dense** agar stabil.
- IndoBERT memakai `eval_strategy` (dengan fallback untuk kompatibilitas versi Transformers).

Catatan: target metrik sebaiknya fokus ke **F1 Fake** dan **Recall Fake**, bukan hanya accuracy, karena dataset imbalanced.


In [5]:
# Install dependencies (jalankan sekali)
#!pip -q install -U pandas numpy scikit-learn matplotlib seaborn transformers datasets accelerate evaluate tensorflow

In [6]:
#pip uninstall -y torch torchvision torchaudio

In [7]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [8]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)
print("CUDA:", torch.cuda.is_available())


2.7.1+cu118
0.22.1+cu118
CUDA: True


In [ ]:
#   !pip show torch torchvision torchaudio transformers

In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix


## 1) Load dataset_final.csv (gold + pseudo)
Kolom penting: `text_final`, `Label` (gold), `label_final` (pseudo), `confidence`.


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("dataset_final.csv")

if "text_final" not in df.columns:
    raise ValueError("Kolom text_final tidak ditemukan.")
if "label_final" not in df.columns:
    raise ValueError("Kolom label_final tidak ditemukan.")

df["text_final"] = df["text_final"].astype(str)

def normalize_label_series(s: pd.Series) -> pd.Series:
    s = s.copy()
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(int)

    s2 = s.astype(str).str.strip().str.lower()
    mapping = {
        "real": 0, "fake": 1,
        "0": 0, "1": 1,
        "false": 0, "true": 1
    }
    s2 = s2.map(mapping)
    return s2  # biarkan NaN untuk difilter

df["y"] = normalize_label_series(df["label_final"])

# Ambil SEMUA baris yang labelnya valid (0/1) -> "pakai semuanya"
df = df[df["y"].isin([0, 1])].copy()
df["y"] = df["y"].astype(int)
df["text"] = df["text_final"]

print("All labeled rows:", len(df))
print(df["y"].value_counts())

# Split sama seperti DNN/ResDNN/WideDeep
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["y"], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

All labeled rows: 7875
y
0    6533
1    1342
Name: count, dtype: int64


In [12]:
print(df["label_final"].astype(str).str.strip().value_counts().head(20))

label_final
Real    6533
Fake    1342
Name: count, dtype: int64


In [13]:
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Train dist:\n", train_df["y"].value_counts(normalize=True))
print("Val dist:\n", val_df["y"].value_counts(normalize=True))
print("Test dist:\n", test_df["y"].value_counts(normalize=True))

Train: 5512 Val: 1181 Test: 1182
Train dist:
 y
0    0.829644
1    0.170356
Name: proportion, dtype: float64
Val dist:
 y
0    0.829805
1    0.170195
Name: proportion, dtype: float64
Test dist:
 y
0    0.829103
1    0.170897
Name: proportion, dtype: float64


## 3) Add pseudo-label (balanced)
Tujuan: tambahkan pseudo-label confidence tinggi **tanpa** mengubah proporsi kelas train terlalu ekstrem.


In [14]:
USE_PSEUDO = False  # tidak dipakai karena sudah all-labeled

train_df = train_df[["text_final", "y"]].copy()

print("Final train size:", len(train_df))
print("Final train dist (y):")
print(train_df["y"].value_counts())


Final train size: 5512
Final train dist (y):
y
0    4573
1     939
Name: count, dtype: int64


## 4) Baselines (TF-IDF) + quick tuning
- LinearSVC biasanya kuat untuk TF-IDF.
- LogisticRegression juga kuat.
- NB sering kalah untuk dataset ini; tetap dicoba dengan alpha.


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

X_train, y_train = train_df['text_final'], train_df['y']
X_test, y_test = test_df['text_final'], test_df['y']

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

c_list = [0.5, 1.0, 2.0]

best_svm = None
best_svm_f1 = -1

for C in c_list:
    pipe = Pipeline([
        ('tfidf', vectorizer),
        ('clf', LinearSVC(class_weight='balanced', C=C))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(test_df['text_final'])
    rep = classification_report(y_test, pred, target_names=['Real','Fake'], digits=4, output_dict=True)
    f1_fake = rep['Fake']['f1-score']
    print(f'LinearSVC C={C} | f1_fake={f1_fake:.4f} | acc={rep["accuracy"]:.4f}')
    if f1_fake > best_svm_f1:
        best_svm_f1 = f1_fake
        best_svm = pipe

# Logistic Regression tuning (C)
best_lr = None
best_lr_f1 = -1

for C in c_list:
    pipe = Pipeline([
        ('tfidf', vectorizer),
        ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', C=C))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(test_df['text_final'])
    rep = classification_report(y_test, pred, target_names=['Real','Fake'], digits=4, output_dict=True)
    f1_fake = rep['Fake']['f1-score']
    print(f'LogReg C={C} | f1_fake={f1_fake:.4f} | acc={rep["accuracy"]:.4f}')
    if f1_fake > best_lr_f1:
        best_lr_f1 = f1_fake
        best_lr = pipe

# Naive Bayes tuning (alpha)
alphas = [0.1, 0.5, 1.0]

best_nb = None
best_nb_f1 = -1
for a in alphas:
    pipe = Pipeline([
        ('tfidf', vectorizer),
        ('clf', MultinomialNB(alpha=a))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(test_df['text_final'])
    rep = classification_report(y_test, pred, target_names=['Real','Fake'], digits=4, output_dict=True)
    f1_fake = rep['Fake']['f1-score']
    print(f'NB alpha={a} | f1_fake={f1_fake:.4f} | acc={rep["accuracy"]:.4f}')
    if f1_fake > best_nb_f1:
        best_nb_f1 = f1_fake
        best_nb = pipe

baseline_models = {
    'tfidf_linearsvm_best': best_svm,
    'tfidf_logreg_best': best_lr,
    'tfidf_nb_best': best_nb
}

for name, model in baseline_models.items():
    pred = model.predict(X_test)
    print('===', name, '===')
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=['Real','Fake'], digits=4))


LinearSVC C=0.5 | f1_fake=0.5589 | acc=0.8384
LinearSVC C=1.0 | f1_fake=0.5476 | acc=0.8350
LinearSVC C=2.0 | f1_fake=0.5300 | acc=0.8274
LogReg C=0.5 | f1_fake=0.5701 | acc=0.8418
LogReg C=1.0 | f1_fake=0.5682 | acc=0.8393
LogReg C=2.0 | f1_fake=0.5657 | acc=0.8350
NB alpha=0.1 | f1_fake=0.4784 | acc=0.8672
NB alpha=0.5 | f1_fake=0.2489 | acc=0.8519
NB alpha=1.0 | f1_fake=0.1209 | acc=0.8401
=== tfidf_linearsvm_best ===
[[870 110]
 [ 81 121]]
              precision    recall  f1-score   support

        Real     0.9148    0.8878    0.9011       980
        Fake     0.5238    0.5990    0.5589       202

    accuracy                         0.8384      1182
   macro avg     0.7193    0.7434    0.7300      1182
weighted avg     0.8480    0.8384    0.8426      1182

=== tfidf_logreg_best ===
[[871 109]
 [ 78 124]]
              precision    recall  f1-score   support

        Real     0.9178    0.8888    0.9031       980
        Fake     0.5322    0.6139    0.5701       202

    accuracy

## 5) TensorFlow models (DNN / Wide&Deep / ResDNN)
Menggunakan TF-IDF **dense**.


In [16]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.optimizers import Adam

print('TF version:', tf.__version__)
print('Num GPUs (TF):', len(tf.config.list_physical_devices('GPU')))


TF version: 2.20.0
Num GPUs (TF): 0


In [ ]:
# Build TF-IDF dense matrices for Keras
from sklearn.feature_extraction.text import TfidfVectorizer

keras_vec = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vec = keras_vec.fit_transform(train_df['text_final'])
X_val_vec = keras_vec.transform(val_df['text_final'])
X_test_vec = keras_vec.transform(test_df['text_final'])

X_train_dense = X_train_vec.toarray().astype('float32')
X_val_dense = X_val_vec.toarray().astype('float32')
X_test_dense = X_test_vec.toarray().astype('float32')

y_train = train_df['y'].values
y_val = val_df['y'].values
y_test = test_df['y'].values

input_dim = X_train_dense.shape[1]
num_classes = 2

print('Dense shapes:', X_train_dense.shape, X_val_dense.shape, X_test_dense.shape)


Dense shapes: (5512, 9236) (1181, 9236) (1182, 9236)


## 6) IndoBERT (Transformers)


In [18]:
import torchaudio, torch
print("torchaudio:", torchaudio.__version__)
print("torch:", torch.__version__)


torchaudio: 2.7.1+cu118
torch: 2.7.1+cu118


In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print("Transformers AutoModel OK")

Transformers AutoModel OK


In [21]:
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tetap pakai text_final kalau di DF kamu memang namanya itu
train_ds = Dataset.from_pandas(train_df[["text_final", "y"]].rename(columns={"text_final": "text", "y": "labels"}))
val_ds   = Dataset.from_pandas(val_df[["text_final", "y"]].rename(columns={"text_final": "text", "y": "labels"}))
test_ds  = Dataset.from_pandas(test_df[["text_final", "y"]].rename(columns={"text_final": "text", "y": "labels"}))

def tok(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

model = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p1", num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    return {"accuracy": acc, "precision_fake": p, "recall_fake": r, "f1_fake": f1}

kwargs = dict(
    output_dir="indobert_runs",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_fake",
    greater_is_better=True,
    logging_steps=50,
)

try:
    args = TrainingArguments(**kwargs, eval_strategy="epoch")
except TypeError:
    args = TrainingArguments(**kwargs, evaluation_strategy="epoch")

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print("=== IndoBERT (test label_final / all-labeled) ===")
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["Real", "Fake"], digits=4))


CUDA available: True
GPU: NVIDIA GeForce RTX 2060


Map: 100%|██████████| 1182/1182 [00:00<00:00, 13190.13 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Fake,Recall Fake,F1 Fake
1,0.335700,0.279145,0.901778,0.814815,0.547264,0.654762
2,0.288100,0.265661,0.901778,0.715736,0.701493,0.708543
3,0.205400,0.354035,0.885690,0.666667,0.656716,0.661654
4,0.154500,0.410673,0.895851,0.675676,0.746269,0.709220
5,0.074400,0.531904,0.891617,0.669767,0.716418,0.692308
6,0.023500,0.581873,0.893311,0.663755,0.756219,0.706977
7,0.038700,0.612989,0.903472,0.720812,0.706468,0.713568
8,0.019700,0.662033,0.903472,0.718593,0.711443,0.715000
9,0.006900,0.714377,0.901778,0.709360,0.716418,0.712871
10,0.020100,0.711881,0.907705,0.734694,0.716418,0.725441


=== IndoBERT (test label_final / all-labeled) ===
[[928  52]
 [ 50 152]]
              precision    recall  f1-score   support

        Real     0.9489    0.9469    0.9479       980
        Fake     0.7451    0.7525    0.7488       202

    accuracy                         0.9137      1182
   macro avg     0.8470    0.8497    0.8483      1182
weighted avg     0.9141    0.9137    0.9139      1182

